# 03 — Partisan lean

Build a master table of presidential results by congressional district, one row per
`(dist_map_year, pres_year, state, district)`:

`dist_map_year, pres_year, state, district, dem_pct, gop_pct, dem_pct_2p_norm, gop_pct_2p_norm, margin_raw, margin_2p_norm, source, quality_flag`

* `margin_raw = dem_pct - gop_pct` (Democratic margin positive); `margin_2p_norm` is the same margin as a share of the two-party total.
* `dem_pct_2p_norm = 100·dem/(dem+gop)` and `gop_pct_2p_norm` are the two-party shares, so `margin_2p_norm = dem_pct_2p_norm − gop_pct_2p_norm`.
* `district` follows the repo convention: two-letter `state`, `0` for at-large, `1..n` otherwise.
* A district's result for a given `pres_year` is repeated for every `dist_map_year` that used the same lines.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

RAW_DIR = Path("../data/raw/pvi.sources")
REFERENCE_DIR = Path("../data/reference")
INTERIM_DIR = Path("../data/interim")

COLUMNS = [
    "dist_map_year", "pres_year", "state", "district", "dem_pct", "gop_pct", "dem_pct_2p_norm", "gop_pct_2p_norm",
    "margin_raw", "margin_2p_norm", "source", "quality_flag",
]
KEY = ["dist_map_year", "pres_year", "state", "district"]
DISTRICT_ORDER = ["dist_map_year", "state", "district", "pres_year"]
MAX_MARGIN_SHIFT = 15  # pct-pts of margin_raw between a district's consecutive pres_years

## Helpers

Each source file gets its own constants and a `PRES_COLUMNS` mapping from `pres_year` to the (dem, gop) columns holding that
year's result. `reshape_wide` selects only those columns and repeats each result for every `dist_map_year` the lines were used in;
`check_selection` confirms the output covers exactly that `dist_map_year` × `pres_year` grid with the right columns in each cell.

In [2]:
def reshape_wide(
    wide: pd.DataFrame,
    pres_columns: dict[int, tuple[str, str]],
    map_years: list[int],
    source: str,
    quality_flag: str,
    missing_states: dict[int, list[str]] | None = None,
) -> pd.DataFrame:
    """Reshape a wide (state, district, <dem col>, <gop col>, ...) frame into one row per master key.

    pres_columns maps each pres_year to the (dem, gop) columns holding that year's result; each result is then
    repeated for every map year in map_years, giving the full map_years x pres_columns grid. missing_states lists,
    per pres_year, the states the source has no result for; those rows are dropped, and any other blank is an error.
    """
    missing = [col for cols in pres_columns.values() for col in cols if col not in wide.columns]
    if missing:
        raise ValueError(f"Missing columns for the requested pres_years: {missing}")
    by_pres_year = pd.concat(
        wide[["state", "district", dem_col, gop_col]]
        .set_axis(["state", "district", "dem_pct", "gop_pct"], axis=1)
        .assign(pres_year=pres_year)
        for pres_year, (dem_col, gop_col) in pres_columns.items()
    )
    blank = by_pres_year[["dem_pct", "gop_pct"]].isna().any(axis=1)
    found = by_pres_year[blank].groupby("pres_year")["state"].agg(lambda s: sorted(set(s))).to_dict()
    documented = {year: sorted(states) for year, states in (missing_states or {}).items()}
    if found != documented:
        raise ValueError(f"Blank results by pres_year {found} don't match missing_states {documented}")
    partial = by_pres_year[~blank].merge(by_pres_year[blank][["pres_year", "state"]].drop_duplicates())
    if not partial.empty:
        raise ValueError(f"States in missing_states that do have some results:\n{partial}")
    by_pres_year = by_pres_year[~blank]
    maps = pd.DataFrame({"dist_map_year": map_years})
    return by_pres_year.merge(maps, how="cross").assign(source=source, quality_flag=quality_flag)


def check_selection(wide: pd.DataFrame, long: pd.DataFrame, pres_columns: dict[int, tuple[str, str]], map_years: list[int]) -> None:
    """Check long covers exactly map_years x pres_years, and each pres_year's rows hold that year's columns."""
    grid = {(m, p) for m in map_years for p in pres_columns}
    found = set(long[["dist_map_year", "pres_year"]].itertuples(index=False, name=None))
    if found != grid:
        raise ValueError(f"(dist_map_year, pres_year) grid mismatch: missing {sorted(grid - found)}, extra {sorted(found - grid)}")
    by_district = wide.set_index(["state", "district"]).sort_index()
    for (map_year, pres_year), rows in long.groupby(["dist_map_year", "pres_year"]):
        got = rows.set_index(["state", "district"])[["dem_pct", "gop_pct"]].sort_index()
        expected = by_district[list(pres_columns[pres_year])].set_axis(["dem_pct", "gop_pct"], axis=1).dropna()
        if not got.equals(expected):
            raise ValueError(f"({map_year}, {pres_year}) rows don't match columns {pres_columns[pres_year]}")


def read_dke(path: Path, district_col: str = "CD") -> pd.DataFrame:
    """Read a Daily Kos Elections district CSV, parsing district_col (`ST-NN` / `ST-AL`) into state and district."""
    wide = pd.read_csv(path, encoding="utf-8-sig")  # the files start with a byte-order mark
    parts = wide[district_col].str.extract(r"^([A-Z]{2})-(AL|\d{2})$")
    if parts.isna().any().any():
        raise ValueError(f"Unexpected district codes: {wide.loc[parts.isna().any(axis=1), district_col].tolist()}")
    wide["state"] = parts[0]
    wide["district"] = parts[1].replace("AL", "0").astype(int)
    return wide


def override_source(long: pd.DataFrame, states_by_year: dict[int, list[str]], source: str) -> pd.DataFrame:
    """Cite `source` on the rows whose (pres_year, state) is listed in states_by_year."""
    keys = {(year, state) for year, states in states_by_year.items() for state in states}
    hit = [(year, state) in keys for year, state in zip(long["pres_year"], long["state"])]
    return long.assign(source=long["source"].mask(hit, source))

## Source: Daily Kos Elections, `06-08-10d_08p.csv`

```
CD,    Incumbent,  Party, dem_08, gop_08, dem_04, gop_04, dem_00, gop_00
AK-AL, "Young, Don", (R), 38,     59,     36,     61,     28,     59
```

* One wide row per district on the 2002-decade lines as used in 2006–2010; `CD` is `ST-NN`, with `ST-AL` for at-large.
* Integer percentages, so `dem + gop` can reach 101 from rounding (2000 sums run low because of Nader).
* Each row is valid for `dist_map_year` {2006, 2008, 2010} × `pres_year` {2000, 2004, 2008}: `pres_year` 2000 comes from
  `dem_00`/`gop_00`, 2004 from `dem_04`/`gop_04`, and 2008 from `dem_08`/`gop_08`.
* `Incumbent` / `Party` are the 2011 roster and are dropped; they are not part of the lean.

In [3]:
FILE_DKE_06D = "06-08-10d_08p.csv"
SOURCE_DKE_06D = "https://docs.google.com/spreadsheets/d/1l7W130tPRF6dQ4JoqlQJSJexnEg4rZct-7kwDXqgoLE"  # Daily Kos Elections
QUALITY_DKE_06D = "approximate"  # integer-rounded percentages
# every district result is valid on all three map years' (identical) lines, for each of the three pres_years
MAP_YEARS_DKE_06D = [2006, 2008, 2010]
PRES_COLUMNS_DKE_06D = {
    2000: ("dem_00", "gop_00"),
    2004: ("dem_04", "gop_04"),
    2008: ("dem_08", "gop_08"),
}

wide_dke_06d = read_dke(RAW_DIR / FILE_DKE_06D)
raw_dke_06d = reshape_wide(wide_dke_06d, PRES_COLUMNS_DKE_06D, MAP_YEARS_DKE_06D, SOURCE_DKE_06D, QUALITY_DKE_06D)
check_selection(wide_dke_06d, raw_dke_06d, PRES_COLUMNS_DKE_06D, MAP_YEARS_DKE_06D)
pd.crosstab(raw_dke_06d["dist_map_year"], raw_dke_06d["pres_year"])  # districts per (map, pres) cell

pres_year,2000,2004,2008
dist_map_year,,,
2006,435,435,435
2008,435,435,435
2010,435,435,435


## Source: Daily Kos Elections, `12-14d_08-12p.csv`

```
CD,    Incumbent,    Party, dem_2012, gop_2012, dem_2008, gop_2008
AK-AL, "Young, Don", (R),   41.2,     55.3,     38.1,     59.7
```

* One wide row per district on the post-2010-census lines used in 2012 and 2014; same `CD` format as above.
* Percentages to one decimal place.
* Each row is valid for `dist_map_year` {2012, 2014} × `pres_year` {2008, 2012}: `pres_year` 2008 comes from
  `dem_2008`/`gop_2008` (Obama–McCain recomputed on the new lines), and 2012 from `dem_2012`/`gop_2012`.

In [4]:
FILE_DKE_12D = "12-14d_08-12p.csv"
SOURCE_DKE_12D = "https://docs.google.com/spreadsheets/d/1xn6nCNM97oFDZ4M-HQgoUT3X4paOiSDsRMSuxbaOBdg/"  # Daily Kos Elections
QUALITY_DKE_12D = "approximate"  # Daily Kos district estimates, rounded to 0.1 pts
MAP_YEARS_DKE_12D = [2012, 2014]
PRES_COLUMNS_DKE_12D = {
    2008: ("dem_2008", "gop_2008"),
    2012: ("dem_2012", "gop_2012"),
}

wide_dke_12d = read_dke(RAW_DIR / FILE_DKE_12D)
raw_dke_12d = reshape_wide(wide_dke_12d, PRES_COLUMNS_DKE_12D, MAP_YEARS_DKE_12D, SOURCE_DKE_12D, QUALITY_DKE_12D)
check_selection(wide_dke_12d, raw_dke_12d, PRES_COLUMNS_DKE_12D, MAP_YEARS_DKE_12D)
pd.crosstab(raw_dke_12d["dist_map_year"], raw_dke_12d["pres_year"])  # districts per (map, pres) cell

pres_year,2008,2012
dist_map_year,,
2012,435,435
2014,435,435


## Source: Daily Kos Elections, `16d_08-12-16p-recon.csv`

```
CD,    Incumbent,    Party, dem_2016, gop_2016, dem_2012, gop_2012, dem_2008, gop_2008
AK-AL, "Young, Don", (R),   37.6,     52.8,     41.2,     55.3,     38.1,     59.7
```

* One wide row per district on the lines used in 2016, after mid-decade redraws in FL, NC and VA; same `CD` format as above.
* Percentages to one decimal place.
* Each row is valid for `dist_map_year` {2016} × `pres_year` {2008, 2012, 2016}: `pres_year` 2008 comes from
  `dem_2008`/`gop_2008`, 2012 from `dem_2012`/`gop_2012`, and 2016 from `dem_2016`/`gop_2016`.
* A reconstruction of Daily Kos's `16d_08-12-16p` file: the original leaves 2012 blank for FL and 2008 blank for FL and VA;
  this version fills them in from a separate sheet (`SOURCE_RECON_16D`), so every district has all three years. Those rows
  carry that sheet as their `source`; everything else is cited to Daily Kos.

In [5]:
FILE_DKE_16D = "16d_08-12-16p-recon.csv"
SOURCE_DKE_16D = "https://docs.google.com/spreadsheets/d/1VfkHtzBTP5gf4jAu8tcVQgsBJ1IDvXEHjuMqYlOgYbA/"  # Daily Kos Elections
# the (pres_year, state) results Daily Kos left blank, filled in from this sheet
SOURCE_RECON_16D = "https://docs.google.com/spreadsheets/d/1oRl7vxEJUUDWJCyrjo62cELJD2ONIVl-D9TSUKiK9jk/edit?gid=1178631925#gid=1178631925"
RECON_STATES_16D = {2008: ["FL", "VA"], 2012: ["FL"]}  # pres_year -> states taken from SOURCE_RECON_16D
QUALITY_DKE_16D = "approximate"  # Daily Kos district estimates, rounded to 0.1 pts
MAP_YEARS_DKE_16D = [2016]
PRES_COLUMNS_DKE_16D = {
    2008: ("dem_2008", "gop_2008"),
    2012: ("dem_2012", "gop_2012"),
    2016: ("dem_2016", "gop_2016"),
}

wide_dke_16d = read_dke(RAW_DIR / FILE_DKE_16D)
raw_dke_16d = reshape_wide(wide_dke_16d, PRES_COLUMNS_DKE_16D, MAP_YEARS_DKE_16D, SOURCE_DKE_16D, QUALITY_DKE_16D)
check_selection(wide_dke_16d, raw_dke_16d, PRES_COLUMNS_DKE_16D, MAP_YEARS_DKE_16D)

raw_dke_16d = override_source(raw_dke_16d, RECON_STATES_16D, SOURCE_RECON_16D)
pd.crosstab(raw_dke_16d["pres_year"], raw_dke_16d["source"])  # districts per (pres, source) on the single 2016 map

source,https://docs.google.com/spreadsheets/d/1VfkHtzBTP5gf4jAu8tcVQgsBJ1IDvXEHjuMqYlOgYbA/,https://docs.google.com/spreadsheets/d/1oRl7vxEJUUDWJCyrjo62cELJD2ONIVl-D9TSUKiK9jk/edit?gid=1178631925#gid=1178631925
pres_year,,
2008,397,38
2012,408,27
2016,435,0


## Source: Daily Kos Elections, `18d_08-12-16p.csv`

```
CD,    Incumbent,    Party, dem_16, gop_16, dem_12, gop_12, dem_08, gop_08, <12 unlabeled helper columns>
AK-AL, "Young, Don", (R),   37.6,   52.8,   41.2,   55.3,   38.1,   59.7,   FALSE, ..., 7
```

* One wide row per district on the lines used in 2018, after PA's court-ordered redraw; same `CD` format as above, but two-digit
  year suffixes. The trailing unlabeled columns are ignored.
* Each row is valid for `dist_map_year` {2018} × `pres_year` {2008, 2012, 2016}: `pres_year` 2008 comes from
  `dem_08`/`gop_08`, 2012 from `dem_12`/`gop_12`, and 2016 from `dem_16`/`gop_16`.
* FL 2008/2012 are filled in the CSV (Daily Kos had none) and match the 2016 reconstruction exactly, so they cite that sheet.
* VA 2008 is blank, but VA's 2018 lines are its 2016 lines, so it is taken from the 2016 file (also citing the reconstruction).
* PA 2008 is blank and has no source on the new lines, so those rows are left out as a known gap.

In [6]:
FILE_DKE_18D = "18d_08-12-16p.csv"
SOURCE_DKE_18D = "https://docs.google.com/spreadsheets/d/1zLNAuRqPauss00HDz4XbTH2HqsCzMe0pR8QmD1K8jk8/"  # Daily Kos Elections
QUALITY_DKE_18D = "approximate"  # Daily Kos district estimates, rounded to 0.1 pts
MAP_YEARS_DKE_18D = [2018]
PRES_COLUMNS_DKE_18D = {
    2008: ("dem_08", "gop_08"),
    2012: ("dem_12", "gop_12"),
    2016: ("dem_16", "gop_16"),
}
RECON_STATES_18D = {2008: ["FL", "VA"], 2012: ["FL"]}  # pres_year -> states citing SOURCE_RECON_16D
MISSING_DKE_18D = {2008: ["PA"]}  # pres_year -> states with no result on the 2018 lines

wide_dke_18d = read_dke(RAW_DIR / FILE_DKE_18D)

# VA's 2018 lines are its 2016 lines: confirm via identical 2012/2016 results, then take its blank 2008 from the 2016 file
va = wide_dke_18d["state"] == "VA"
va_16d = wide_dke_16d.set_index("CD").loc[wide_dke_18d.loc[va, "CD"]]
if not wide_dke_18d.loc[va, ["dem_08", "gop_08"]].isna().all().all():
    raise ValueError("Expected VA 2008 to be blank in the 2018 file")
if not (wide_dke_18d.loc[va, ["dem_12", "gop_12", "dem_16", "gop_16"]].to_numpy()
        == va_16d[["dem_2012", "gop_2012", "dem_2016", "gop_2016"]].to_numpy()).all():
    raise ValueError("VA 2012/2016 results differ between the 2016 and 2018 files, so its lines may have changed")
wide_dke_18d.loc[va, ["dem_08", "gop_08"]] = va_16d[["dem_2008", "gop_2008"]].to_numpy()

raw_dke_18d = reshape_wide(
    wide_dke_18d, PRES_COLUMNS_DKE_18D, MAP_YEARS_DKE_18D, SOURCE_DKE_18D, QUALITY_DKE_18D, MISSING_DKE_18D
)
check_selection(wide_dke_18d, raw_dke_18d, PRES_COLUMNS_DKE_18D, MAP_YEARS_DKE_18D)
raw_dke_18d = override_source(raw_dke_18d, RECON_STATES_18D, SOURCE_RECON_16D)
pd.crosstab(raw_dke_18d["pres_year"], raw_dke_18d["source"])  # districts per (pres, source) on the single 2018 map

source,https://docs.google.com/spreadsheets/d/1oRl7vxEJUUDWJCyrjo62cELJD2ONIVl-D9TSUKiK9jk/edit?gid=1178631925#gid=1178631925,https://docs.google.com/spreadsheets/d/1zLNAuRqPauss00HDz4XbTH2HqsCzMe0pR8QmD1K8jk8/
pres_year,,
2008,38,379
2012,27,408
2016,0,435


## Source: Daily Kos Elections, `20d_08-12-16-20p.csv`

```
District, Incumbent,    Party, 2020_dem, 2020_gop, 2016_dem, 2016_gop, 2012_dem, 2012_gop, Obama, McCain
AK-AL,    Mary Peltola, (D),   43.0,     53.1,     37.6,     52.8,     41.2,     55.3,     38.1,  59.7
```

* One wide row per district on the lines used in 2020, after NC's court-ordered redraw. Same `ST-NN` codes as above, but in a
  `District` column, with year-first column names.
* Each row is valid for `dist_map_year` {2020} × `pres_year` {2012, 2016, 2020}: `pres_year` 2012 comes from
  `2012_dem`/`2012_gop`, 2016 from `2016_dem`/`2016_gop`, and 2020 from `2020_dem`/`2020_gop`. The 2008 `Obama`/`McCain` columns are not used.
* FL 2012 is filled in the CSV (Daily Kos had none) and matches the 2016 reconstruction exactly, so it cites that sheet.

In [7]:
FILE_DKE_20D = "20d_08-12-16-20p.csv"
SOURCE_DKE_20D = "https://docs.google.com/spreadsheets/d/1XbUXnI9OyfAuhP5P3vWtMuGc5UJlrhXbzZo3AwMuHtk/"  # Daily Kos Elections
QUALITY_DKE_20D = "approximate"  # Daily Kos district estimates, rounded to 0.1 pts
MAP_YEARS_DKE_20D = [2020]
PRES_COLUMNS_DKE_20D = {
    2012: ("2012_dem", "2012_gop"),
    2016: ("2016_dem", "2016_gop"),
    2020: ("2020_dem", "2020_gop"),
}
RECON_STATES_20D = {2012: ["FL"]}  # pres_year -> states citing SOURCE_RECON_16D

wide_dke_20d = read_dke(RAW_DIR / FILE_DKE_20D, district_col="District")
raw_dke_20d = reshape_wide(wide_dke_20d, PRES_COLUMNS_DKE_20D, MAP_YEARS_DKE_20D, SOURCE_DKE_20D, QUALITY_DKE_20D)
check_selection(wide_dke_20d, raw_dke_20d, PRES_COLUMNS_DKE_20D, MAP_YEARS_DKE_20D)
raw_dke_20d = override_source(raw_dke_20d, RECON_STATES_20D, SOURCE_RECON_16D)
pd.crosstab(raw_dke_20d["pres_year"], raw_dke_20d["source"])  # districts per (pres, source) on the single 2020 map

source,https://docs.google.com/spreadsheets/d/1XbUXnI9OyfAuhP5P3vWtMuGc5UJlrhXbzZo3AwMuHtk/,https://docs.google.com/spreadsheets/d/1oRl7vxEJUUDWJCyrjo62cELJD2ONIVl-D9TSUKiK9jk/edit?gid=1178631925#gid=1178631925
pres_year,,
2012,408,27
2016,435,0
2020,435,0


## Source: Daily Kos Elections, `22d_20p.csv`

```
District, Incumbent,    Party, dem_2020, gop_2020
AK-AL,    Mary Peltola, (D),   43.0,     53.1
```

* One wide row per district on the post-2020-census lines used in 2022. Same `District` column and `ST-NN` codes as the 2020 file.
* Each row is valid for `dist_map_year` {2022} × `pres_year` {2020}: `pres_year` 2020 comes from `dem_2020`/`gop_2020`.

In [8]:
FILE_DKE_22D = "22d_20p.csv"
SOURCE_DKE_22D = "https://docs.google.com/spreadsheets/d/1CKngqOp8fzU22JOlypoxNsxL6KSAH920Whc-rd7ebuM/"  # Daily Kos Elections
QUALITY_DKE_22D = "approximate"  # Daily Kos district estimates, rounded to 0.1 pts
MAP_YEARS_DKE_22D = [2022]
PRES_COLUMNS_DKE_22D = {2020: ("dem_2020", "gop_2020")}

wide_dke_22d = read_dke(RAW_DIR / FILE_DKE_22D, district_col="District")
raw_dke_22d = reshape_wide(wide_dke_22d, PRES_COLUMNS_DKE_22D, MAP_YEARS_DKE_22D, SOURCE_DKE_22D, QUALITY_DKE_22D)
check_selection(wide_dke_22d, raw_dke_22d, PRES_COLUMNS_DKE_22D, MAP_YEARS_DKE_22D)
pd.crosstab(raw_dke_22d["dist_map_year"], raw_dke_22d["pres_year"])  # districts per (map, pres) cell

pres_year,2020
dist_map_year,
2022,435


## Source: ALARM atlas, `2024-atlas.csv`

```
plan_id,                  cycle, state, district, district_id, pres_2008_dem_share, ..., pres_2024_rep_share, pres_2008_dem_votes, ...
AL_us_house_2024_enacted, 2024,  AL,    1,        AL-01,       0.2778,              ..., 0.7729,              77671,               ...
```

* One row per district on each state's enacted 2024 map; `district` is already numeric, but at-large states use `1`, converted to `0`.
* Shares are **two-party** fractions (`dem + rep = 1`), scaled to percentages here. So `dem_pct + gop_pct = 100` and
  `margin_raw` equals `margin_2p_norm` for these rows, unlike the Daily Kos rows; `quality_flag = "two_party"` marks this.
* Each row is valid for `dist_map_year` {2024} × `pres_year` {2016, 2020, 2024}: `pres_year` 2016 comes from
  `pres_2016_dem_share`/`pres_2016_rep_share`, and likewise for 2020 and 2024. The 2008/2012 shares and all `_votes` columns are not used.

In [9]:
FILE_ATLAS_24D = "2024-atlas.csv"
SOURCE_ATLAS_24D = "https://christophertkenny.com/atlas"
QUALITY_ATLAS_24D = "two_party"  # dem/gop are two-party shares, so third-party votes are excluded
MAP_YEARS_ATLAS_24D = [2024]
PRES_COLUMNS_ATLAS_24D = {
    2016: ("pres_2016_dem_share", "pres_2016_rep_share"),
    2020: ("pres_2020_dem_share", "pres_2020_rep_share"),
    2024: ("pres_2024_dem_share", "pres_2024_rep_share"),
}


def read_atlas(path: Path, pres_columns: dict[int, tuple[str, str]]) -> pd.DataFrame:
    """Read an atlas district CSV: check it is one enacted map, set at-large districts to 0, and scale the
    (dem, rep) share columns named in pres_columns from fractions to percent."""
    wide = pd.read_csv(path)
    expected_plans = wide["state"] + "_us_house_" + wide["cycle"].astype(str) + "_enacted"
    if not (wide["plan_id"] == expected_plans).all() or wide["cycle"].nunique() != 1:
        raise ValueError(f"Expected one enacted plan per state for a single cycle; got {sorted(wide['plan_id'].unique())}")
    at_large = wide.groupby("state")["district"].transform("size") == 1
    if not (wide.loc[at_large, "district"] == 1).all():
        raise ValueError("Expected at-large states to be numbered district 1")
    wide["district"] = wide["district"].mask(at_large, 0)
    shares = [col for cols in pres_columns.values() for col in cols]
    wide[shares] = 100 * wide[shares]
    return wide


wide_atlas_24d = read_atlas(RAW_DIR / FILE_ATLAS_24D, PRES_COLUMNS_ATLAS_24D)
raw_atlas_24d = reshape_wide(wide_atlas_24d, PRES_COLUMNS_ATLAS_24D, MAP_YEARS_ATLAS_24D, SOURCE_ATLAS_24D, QUALITY_ATLAS_24D)
check_selection(wide_atlas_24d, raw_atlas_24d, PRES_COLUMNS_ATLAS_24D, MAP_YEARS_ATLAS_24D)
pd.crosstab(raw_atlas_24d["dist_map_year"], raw_atlas_24d["pres_year"])  # districts per (map, pres) cell

pres_year,2016,2020,2024
dist_map_year,,,
2024,435,435,435


## Master table

In [10]:
def add_margins(df: pd.DataFrame) -> pd.DataFrame:
    """Add two-party shares, the D-R margin, and the margin as a share of the two-party total."""
    out = df.copy()
    # a zero two-party total has no defined share, so it becomes NaN instead of inf
    two_party = (out["dem_pct"] + out["gop_pct"]).where(lambda t: t > 0)
    out["dem_pct_2p_norm"] = 100 * out["dem_pct"] / two_party
    out["gop_pct_2p_norm"] = 100 * out["gop_pct"] / two_party
    out["margin_raw"] = out["dem_pct"] - out["gop_pct"]
    out["margin_2p_norm"] = 100 * out["margin_raw"] / two_party
    return out


def margin_shift(df: pd.DataFrame) -> pd.Series:
    """Change in margin_raw from the same district's previous pres_year on the same map (NaN for the first)."""
    ordered = df.sort_values(DISTRICT_ORDER)
    return ordered.groupby(["dist_map_year", "state", "district"])["margin_raw"].diff().reindex(df.index)


def build_master(sources: list[pd.DataFrame]) -> pd.DataFrame:
    """Combine per-source frames (each carrying its own source and quality_flag) and add margins."""
    master = add_margins(pd.concat(sources, ignore_index=True))
    dupes = master.duplicated(KEY, keep=False)
    if dupes.any():
        raise ValueError(f"Duplicate keys across sources:\n{master.loc[dupes, KEY + ['source']]}")
    return master[COLUMNS].sort_values(DISTRICT_ORDER, ignore_index=True)


master = build_master([raw_dke_06d, raw_dke_12d, raw_dke_16d, raw_dke_18d, raw_dke_20d, raw_dke_22d, raw_atlas_24d])
# (dist_map_year, pres_year, state) cells a source documents as unavailable
GAPS = {(m, p, s) for m in MAP_YEARS_DKE_18D for p, states in MISSING_DKE_18D.items() for s in states}
master

,dist_map_year,pres_year,state,district,dem_pct,gop_pct,dem_pct_2p_norm,gop_pct_2p_norm,margin_raw,margin_2p_norm,source,quality_flag
0,2006,2000,AK,0,28.000000,59.000000,32.183908,67.816092,-31.000000,-35.632184,https://docs.google.com/spreadsheets/d/1l7W130...,approximate
1,2006,2004,AK,0,36.000000,61.000000,37.113402,62.886598,-25.000000,-25.773196,https://docs.google.com/spreadsheets/d/1l7W130...,approximate
2,2006,2008,AK,0,38.000000,59.000000,39.175258,60.824742,-21.000000,-21.649485,https://docs.google.com/spreadsheets/d/1l7W130...,approximate
3,2006,2000,AL,1,38.000000,60.000000,38.775510,61.224490,-22.000000,-22.448980,https://docs.google.com/spreadsheets/d/1l7W130...,approximate
4,2006,2004,AL,1,35.000000,64.000000,35.353535,64.646465,-29.000000,-29.292929,https://docs.google.com/spreadsheets/d/1l7W130...,approximate
...,...,...,...,...,...,...,...,...,...,...,...,...
11287,2024,2020,WV,2,31.156811,68.843189,31.156811,68.843189,-37.686378,-37.686378,https://christophertkenny.com/atlas,two_party
11288,2024,2024,WV,2,29.807960,70.192040,29.807960,70.192040,-40.384079,-40.384079,https://christophertkenny.com/atlas,two_party
11289,2024,2016,WY,0,24.294867,75.705133,24.294867,75.705133,-51.410266,-51.410266,https://christophertkenny.com/atlas,two_party
11290,2024,2020,WY,0,27.519674,72.480326,27.519674,72.480326,-44.960652,-44.960652,https://christophertkenny.com/atlas,two_party


## Sanity checks

In [11]:
def check_districts(df: pd.DataFrame, gaps: set[tuple[int, int, str]], path: Path = REFERENCE_DIR / "district_counts.json") -> None:
    """Check every (dist_map_year, pres_year) has exactly the expected districts per state: [0] or 1..n.

    gaps holds the (dist_map_year, pres_year, state) cells a source documents as unavailable; they must be absent.
    """
    with open(path) as f:
        counts = json.load(f)
    map_pres_years = df[["dist_map_year", "pres_year"]].drop_duplicates().itertuples(index=False)
    expected = {
        (map_year, pres_year, state): [0] if n == 1 else list(range(1, n + 1))
        for map_year, pres_year in map_pres_years for state, n in counts[str(map_year)].items()
        if (map_year, pres_year, state) not in gaps
    }
    found = df.groupby(["dist_map_year", "pres_year", "state"])["district"].agg(sorted).to_dict()
    if found != expected:
        bad = sorted(k for k in found.keys() | expected.keys() if found.get(k) != expected.get(k))
        raise ValueError(f"Districts don't match district_counts.json for (map, pres, state): {bad}")


def check_pcts(df: pd.DataFrame) -> None:
    """Check shares are in [0, 100] and dem + gop never exceeds 100 beyond integer rounding."""
    pcts = df[["dem_pct", "gop_pct"]]
    if pcts.isna().any().any() or not pcts.stack().between(0, 100).all():
        raise ValueError("dem_pct / gop_pct missing or outside [0, 100]")
    if (pcts.sum(axis=1) > 101).any():
        raise ValueError("dem_pct + gop_pct exceeds 101")


check_districts(master, GAPS)
check_pcts(master)
master.assign(two_party=master["dem_pct"] + master["gop_pct"]).groupby("pres_year")["two_party"].describe()

,count,mean,std,min,25%,50%,75%,max
pres_year,,,,,,,,
2000,1305.0,97.183908,2.240156e+00,87.0,96.000,97.0,99.000,101.0
2004,1305.0,99.268966,6.876391e-01,97.0,99.000,99.0,100.000,101.0
2008,3027.0,98.585861,8.201113e-01,95.0,98.000,98.9,99.000,100.0
2012,2175.0,98.373379,6.963727e-01,94.9,97.900,98.5,98.900,100.0
2016,1740.0,96.221897,3.286681e+00,70.5,94.475,96.1,99.025,100.0
2020,1305.0,98.862222,9.472059e-01,95.5,98.100,98.7,100.000,100.0
2024,435.0,100.000000,9.670209e-09,100.0,100.000,100.0,100.000,100.0


### Margin shifts

Flag districts whose `margin_raw` moved more than `MAX_MARGIN_SHIFT` pct-pts from the previous `pres_year`.
This compares raw margins, so it also catches uniform
national swings (2004 → 2008 was ~10 pts nationally, 2008 → 2012 ~3 pts, 2012 → 2016 ~2 pts, 2016 → 2020 ~2 pts, 2020 → 2024 ~6 pts); the `vs_median` column separates the district's own movement.

In [12]:
shifts = (
    master.assign(shift=margin_shift(master))
    .assign(vs_median=lambda d: d["shift"] - d.groupby(["dist_map_year", "pres_year"])["shift"].transform("median"))
    .query("shift.abs() > @MAX_MARGIN_SHIFT")
)
print(shifts.groupby(["dist_map_year", "pres_year"]).size().unstack())
# results are copied across map years (and repeat across files where lines didn't change), so show each once
shifts.drop_duplicates(["pres_year", "state", "district", "dem_pct", "gop_pct"]).sort_values("vs_median", key=abs, ascending=False).head(15)

pres_year      2004  2008  2012  2016  2020  2024
dist_map_year                                    
2006            6.0  83.0   NaN   NaN   NaN   NaN
2008            6.0  83.0   NaN   NaN   NaN   NaN
2010            6.0  83.0   NaN   NaN   NaN   NaN
2012            NaN   NaN   8.0   NaN   NaN   NaN
2014            NaN   NaN   8.0   NaN   NaN   NaN
2016            NaN   NaN   8.0  45.0   NaN   NaN
2018            NaN   NaN   8.0  47.0   NaN   NaN
2020            NaN   NaN   NaN  47.0   7.0   NaN
2024            NaN   NaN   NaN   NaN   6.0  26.0


,dist_map_year,pres_year,state,district,dem_pct,gop_pct,dem_pct_2p_norm,gop_pct_2p_norm,margin_raw,margin_2p_norm,source,quality_flag,shift,vs_median
6854,2016,2016,UT,3,23.300000,47.200000,33.049645,66.950355,-23.90000,-33.900709,https://docs.google.com/spreadsheets/d/1VfkHtz...,approximate,34.900000,36.600000
6857,2016,2016,UT,4,32.400000,39.100000,45.314685,54.685315,-6.70000,-9.370629,https://docs.google.com/spreadsheets/d/1VfkHtz...,approximate,30.300000,32.000000
6848,2016,2016,UT,1,22.400000,49.700000,31.067961,68.932039,-27.30000,-37.864078,https://docs.google.com/spreadsheets/d/1VfkHtz...,approximate,29.700000,31.400000
6566,2016,2016,OH,6,26.700000,69.300000,27.812500,72.187500,-42.60000,-44.375000,https://docs.google.com/spreadsheets/d/1VfkHtz...,approximate,-30.100000,-28.400000
6851,2016,2016,UT,2,32.000000,46.000000,41.025641,58.974359,-14.00000,-17.948718,https://docs.google.com/spreadsheets/d/1VfkHtz...,approximate,24.800000,26.500000
35,2006,2008,AR,4,39.000000,58.000000,40.206186,59.793814,-19.00000,-19.587629,https://docs.google.com/spreadsheets/d/1l7W130...,approximate,-16.000000,-26.000000
26,2006,2008,AR,1,38.000000,59.000000,39.175258,60.824742,-21.00000,-21.649485,https://docs.google.com/spreadsheets/d/1l7W130...,approximate,-16.000000,-26.000000
374,2006,2008,HI,1,70.000000,28.000000,71.428571,28.571429,42.00000,42.857143,https://docs.google.com/spreadsheets/d/1l7W130...,approximate,36.000000,26.000000
377,2006,2008,HI,2,73.000000,25.000000,74.489796,25.510204,48.00000,48.979592,https://docs.google.com/spreadsheets/d/1l7W130...,approximate,36.000000,26.000000
10330,2024,2020,FL,28,46.792630,53.207370,46.792630,53.207370,-6.41474,-6.414740,https://christophertkenny.com/atlas,two_party,-22.470018,-25.573918


## National presidential results

`pres_raw.csv` holds national vote counts, which serve as the baseline for district lean:

```
pres_year, dem_raw,      gop_raw,      other_raw,   total_raw,     source
2000,      "50,999,897", "50,456,002", "3,949,201", "105,405,100", Wikipedia
```

With D, G, O the Democratic, Republican and other vote counts and T = D + G + O:

* `dem_pct = 100·D/T`, `gop_pct = 100·G/T`, `other_pct = 100·O/T` (summing to 100)
* `dem_pct_2p_norm = 100·D/(D+G)`, `gop_pct_2p_norm = 100·G/(D+G)` (summing to 100)
* `margin_raw = dem_pct − gop_pct`, `margin_2p_norm = dem_pct_2p_norm − gop_pct_2p_norm`, the same definitions as the district table
* `dem_votes`, `gop_votes`, `other_votes` are the counts as given (renamed from `*_raw`, since `_votes` means counts in this repo);
  `total_raw` is only used to check that the counts add up.

In [13]:
NATIONAL_COLUMNS = [
    "pres_year", "dem_pct_2p_norm", "gop_pct_2p_norm", "dem_pct", "gop_pct", "other_pct",
    "margin_raw", "margin_2p_norm", "dem_votes", "gop_votes", "other_votes", "source",
]
VOTE_RENAMES = {"dem_raw": "dem_votes", "gop_raw": "gop_votes", "other_raw": "other_votes", "total_raw": "total_votes"}


def load_national(path: Path = RAW_DIR / "pres_raw.csv") -> pd.DataFrame:
    """Read national presidential vote counts as *_votes, checking each year appears once and its counts add up to the total."""
    df = pd.read_csv(path, thousands=",").rename(columns=VOTE_RENAMES)
    counts = df[["dem_votes", "gop_votes", "other_votes"]]
    if df["pres_year"].duplicated().any():
        raise ValueError(f"Duplicate pres_year: {df.loc[df['pres_year'].duplicated(), 'pres_year'].tolist()}")
    if not (counts > 0).all().all():
        raise ValueError("Vote counts must be positive")
    if not (counts.sum(axis=1) == df["total_votes"]).all():
        raise ValueError(f"dem + gop + other != total for {df.loc[counts.sum(axis=1) != df['total_votes'], 'pres_year'].tolist()}")
    return df


def to_national(df: pd.DataFrame) -> pd.DataFrame:
    """Add total and two-party shares and margins, in the national schema."""
    total = df["dem_votes"] + df["gop_votes"] + df["other_votes"]
    two_party = df["dem_votes"] + df["gop_votes"]
    out = df.assign(
        dem_pct=100 * df["dem_votes"] / total,
        gop_pct=100 * df["gop_votes"] / total,
        other_pct=100 * df["other_votes"] / total,
        dem_pct_2p_norm=100 * df["dem_votes"] / two_party,
        gop_pct_2p_norm=100 * df["gop_votes"] / two_party,
    )
    out["margin_raw"] = out["dem_pct"] - out["gop_pct"]
    out["margin_2p_norm"] = out["dem_pct_2p_norm"] - out["gop_pct_2p_norm"]
    return out[NATIONAL_COLUMNS].sort_values("pres_year", ignore_index=True)


presidential = to_national(load_national())
if set(presidential["pres_year"]) != set(master["pres_year"]):
    raise ValueError(f"National years {sorted(presidential['pres_year'])} don't match master's {sorted(master['pres_year'].unique())}")
for cols in (["dem_pct", "gop_pct", "other_pct"], ["dem_pct_2p_norm", "gop_pct_2p_norm"]):
    if not ((presidential[cols].sum(axis=1) - 100).abs() < 1e-9).all():
        raise ValueError(f"{cols} don't sum to 100")

presidential.to_csv(INTERIM_DIR / "presidential.csv", index=False)
presidential

,pres_year,dem_pct_2p_norm,gop_pct_2p_norm,dem_pct,gop_pct,other_pct,margin_raw,margin_2p_norm,dem_votes,gop_votes,other_votes,source
0,2000,50.268045,49.731955,48.384658,47.868653,3.746689,0.516004,0.536090,50999897,50456002,3949201,Wikipedia
1,2004,48.756013,51.243987,48.268983,50.732104,0.998912,-2.463121,-2.487974,59028444,62040610,1221576,Wikipedia
2,2008,53.688848,46.311152,52.925515,45.652714,1.421770,7.272801,7.377695,69498516,59948323,1866981,Wikipedia
3,2012,51.963862,48.036138,51.063707,47.204021,1.732272,3.859686,3.927725,65915795,60933504,2236111,Wikipedia
4,2016,51.113289,48.886711,48.184578,46.085580,5.729842,2.098998,2.226578,65853514,62984828,7830934,Wikipedia
5,2020,52.269835,47.730165,51.305744,46.849806,1.844450,4.455938,4.539670,81283501,74223975,2922155,Wikipedia
6,2024,49.249946,50.750054,48.324165,49.796074,1.879761,-1.471909,-1.500108,75017613,77302580,2918109,Wikipedia


## Backsolving 2016 on the 2022 lines from Cook's 2023 PVI

The 2022 lines have 2020 results but no 2016 results. Cook's public 2023 PVI uses 2016 and 2020 on those lines with the same
formula as `03b` (relative to the national two-party Democratic share `nat(p)`, weighted 0.75 on 2020 and 0.25 on 2016), so the
2016 two-party share can be solved for:

```
rel20           = dem_pct_2p_norm(2022 lines, 2020) − nat(2020)
rel16           = (cook_pvi − 0.75·rel20) / 0.25
dem_pct_2p_norm = nat(2016) + rel16,   gop_pct_2p_norm = 100 − dem_pct_2p_norm,   margin_2p_norm = dem − gop
```

Only two-party quantities can be recovered, so `dem_pct`, `gop_pct` and `margin_raw` are left blank, and `quality_flag = "backsolved"`.
Any gap between Cook's own 2020 district figures and Daily Kos's is multiplied by 3 in `rel16`, so the result is checked against
the atlas's actual 2016 results wherever the 2024 lines are unchanged from 2022.

In [14]:
FILE_COOK_2023 = "cook_pvi_public_2023.csv"
SOURCE_COOK = "https://www.cookpolitical.com/cook-pvi"
QUALITY_BACKSOLVED = "backsolved"
PVI_WEIGHTS_75_25 = (0.75, 0.25)  # (prior, anteprior): Cook since 2021, and 03b
PVI_WEIGHTS_50_50 = (0.5, 0.5)  # Cook's older method


def clean_cook_pvi(path: Path) -> pd.DataFrame:
    """Split `Dist` (`ST-NN` / `ST-AL`) into state and district, and parse `2023 Raw PVI` (`D+x` / `R+x`) to a signed float."""
    cook = pd.read_csv(path, dtype={"2023 Raw PVI": str})
    dist = cook["Dist"].str.extract(r"^([A-Z]{2})-(AL|\d{2})$")
    pvi = cook["2023 Raw PVI"].str.extract(r"^([DR])\+(\d+(?:\.\d+)?)$")
    if dist.isna().any().any() or pvi.isna().any().any():
        bad = cook[dist.isna().any(axis=1) | pvi.isna().any(axis=1)]
        raise ValueError(f"Unexpected Dist / 2023 Raw PVI values:\n{bad}")
    cleaned = cook.drop(columns="Dist").assign(
        state=dist[0],
        district=dist[1].replace("AL", "0").astype(int),
        **{"2023 Raw PVI": pvi[0].map({"D": 1, "R": -1}) * pvi[1].astype(float)},
    )
    return cleaned[["state", "district"] + [c for c in cook.columns if c != "Dist"]]


def backsolve_anteprior(
    pvi: pd.DataFrame,
    lean: pd.DataFrame,
    national: pd.DataFrame,
    map_year: int,
    known: int,
    missing: int,
    weights: tuple[float, float],
) -> pd.DataFrame:
    """Solve the published PVI (weights on known, missing) for the missing year's two-party share on map_year's lines."""
    nat = national.set_index("pres_year")["dem_pct_2p_norm"]
    on_map = lean[(lean["dist_map_year"] == map_year) & (lean["pres_year"] == known)]
    merged = on_map.merge(pvi[["state", "district", "pvi"]], on=["state", "district"], how="outer", validate="one_to_one", indicator=True)
    if not (merged["_merge"] == "both").all():
        raise ValueError(f"PVI and {map_year}/{known} districts don't match:\n{merged[merged['_merge'] != 'both']}")
    rel_known = merged["dem_pct_2p_norm"] - nat[known]
    rel_missing = (merged["pvi"] - weights[0] * rel_known) / weights[1]
    dem_2p = nat[missing] + rel_missing
    out = pd.DataFrame({
        "dist_map_year": map_year, "pres_year": missing, "state": merged["state"], "district": merged["district"],
        "dem_pct": np.nan, "gop_pct": np.nan, "dem_pct_2p_norm": dem_2p, "gop_pct_2p_norm": 100 - dem_2p,
        "margin_raw": np.nan, "margin_2p_norm": 2 * dem_2p - 100,
        "source": SOURCE_COOK, "quality_flag": QUALITY_BACKSOLVED,
    })
    # round trip: the known year plus the backsolved year must reproduce the published PVI exactly
    implied = weights[0] * rel_known + weights[1] * (out["dem_pct_2p_norm"] - nat[missing])
    if not ((implied - merged["pvi"]).abs() < 1e-9).all():
        raise ValueError("Backsolved shares don't reproduce the published PVI")
    if not out["dem_pct_2p_norm"].between(0, 100).all():
        raise ValueError(f"Backsolved shares outside [0, 100]:\n{out[~out['dem_pct_2p_norm'].between(0, 100)]}")
    return out[COLUMNS].sort_values(["state", "district"], ignore_index=True)


cook_2023 = clean_cook_pvi(RAW_DIR / FILE_COOK_2023)
cook_2023.to_csv(INTERIM_DIR / "cook_pvi_public_2023_cleaned.csv", index=False)

backsolved_22d_16p = backsolve_anteprior(
    cook_2023.rename(columns={"2023 Raw PVI": "pvi"}), master, presidential,
    map_year=2022, known=2020, missing=2016, weights=PVI_WEIGHTS_75_25,
)
backsolved_22d_16p

,dist_map_year,pres_year,state,district,dem_pct,gop_pct,dem_pct_2p_norm,gop_pct_2p_norm,margin_raw,margin_2p_norm,source,quality_flag
0,2022,2016,AK,0,NaN,NaN,41.607622,58.392378,NaN,-16.784756,https://www.cookpolitical.com/cook-pvi,backsolved
1,2022,2016,AL,1,NaN,NaN,34.964937,65.035063,NaN,-30.070125,https://www.cookpolitical.com/cook-pvi,backsolved
2,2022,2016,AL,2,NaN,NaN,33.188248,66.811752,NaN,-33.623504,https://www.cookpolitical.com/cook-pvi,backsolved
3,2022,2016,AL,3,NaN,NaN,32.097324,67.902676,NaN,-35.805351,https://www.cookpolitical.com/cook-pvi,backsolved
4,2022,2016,AL,4,NaN,NaN,18.599157,81.400843,NaN,-62.801685,https://www.cookpolitical.com/cook-pvi,backsolved
...,...,...,...,...,...,...,...,...,...,...,...,...
430,2022,2016,WI,7,NaN,NaN,39.147362,60.852638,NaN,-21.705276,https://www.cookpolitical.com/cook-pvi,backsolved
431,2022,2016,WI,8,NaN,NaN,40.886855,59.113145,NaN,-18.226291,https://www.cookpolitical.com/cook-pvi,backsolved
432,2022,2016,WV,1,NaN,NaN,27.287058,72.712942,NaN,-45.425885,https://www.cookpolitical.com/cook-pvi,backsolved
433,2022,2016,WV,2,NaN,NaN,28.360105,71.639895,NaN,-43.279789,https://www.cookpolitical.com/cook-pvi,backsolved


### Validation against the atlas

Where a district's lines didn't change between 2022 and 2024 (its 2020 two-party shares on the two maps agree to within 0.5 pts),
the atlas's actual 2016 result on the 2024 lines should match the backsolved value.

In [15]:
UNCHANGED_TOLERANCE = 0.5  # pts of 2020 dem_pct_2p_norm between the 2022 and 2024 lines

shares = master.pivot_table(index=["state", "district"], columns=["dist_map_year", "pres_year"], values="dem_pct_2p_norm")
validation = backsolved_22d_16p.set_index(["state", "district"])[["dem_pct_2p_norm"]].rename(columns={"dem_pct_2p_norm": "backsolved_2016"})
validation["atlas_2016"] = shares[(2024, 2016)]
validation["error"] = validation["backsolved_2016"] - validation["atlas_2016"]
unchanged = (shares[(2022, 2020)] - shares[(2024, 2020)]).abs() <= UNCHANGED_TOLERANCE
checked = validation[unchanged.reindex(validation.index, fill_value=False)]
print(f"{len(checked)} districts with unchanged lines: mean |error| {checked['error'].abs().mean():.2f} pts, "
      f"median {checked['error'].abs().median():.2f}")
checked[checked["error"].abs() > 1].sort_values("error", key=abs, ascending=False)

390 districts with unchanged lines: mean |error| 0.18 pts, median 0.09


,,backsolved_2016,atlas_2016,error
state,district,,,
NJ,12,53.851183,66.127411,-12.276229
GA,4,76.930155,78.537582,-1.607427
LA,1,26.362916,27.828212,-1.465296
VA,1,41.852134,42.952723,-1.100589
OK,4,32.288367,31.208025,1.080343
TX,22,38.190879,37.185421,1.005457


### Override: NJ-12

Cook's 2023 NJ-12 PVI (D+12.09) conflicts with the results on its own lines. With the 2020 share (67.5% on both maps, and
consistent with Cook's 2025 PVI of D+13), it forces a 2016 share of about 53.9%, while the atlas's observed 2016 share is 66.1%.
NJ-12's lines didn't change for 2024, so the atlas's 2016 result is used instead of the backsolved value. Its row then carries
the atlas's `source` and `quality_flag` (two-party shares, so `dem_pct = dem_pct_2p_norm`), and its 2022 PVI becomes D+15.16.

In [16]:
ATLAS_2016_OVERRIDES = [("NJ", 12)]  # (state, district) whose 2022-lines 2016 result is taken from the unchanged 2024 lines

changed = [key for key in ATLAS_2016_OVERRIDES if not unchanged.get(key, False)]
if changed:
    raise ValueError(f"Lines changed between 2022 and 2024, so the atlas 2016 result can't be used: {changed}")
atlas_2016 = master[(master["dist_map_year"] == 2024) & (master["pres_year"] == 2016)].set_index(["state", "district"])
backsolved_22d_16p = (
    pd.concat([
        backsolved_22d_16p.set_index(["state", "district"]).drop(ATLAS_2016_OVERRIDES),
        atlas_2016.loc[ATLAS_2016_OVERRIDES].assign(dist_map_year=2022),
    ])
    .reset_index()[COLUMNS]
    .sort_values(["state", "district"], ignore_index=True)
)
backsolved_22d_16p.to_csv(INTERIM_DIR / "partisan_lean_2022d_2016p.csv", index=False)
backsolved_22d_16p.set_index(["state", "district"]).loc[ATLAS_2016_OVERRIDES]

,,dist_map_year,pres_year,dem_pct,gop_pct,dem_pct_2p_norm,gop_pct_2p_norm,margin_raw,margin_2p_norm,source,quality_flag
state,district,,,,,,,,,,
NJ,12,2022,2016,66.127411,33.872589,66.127411,33.872589,32.254823,32.254823,https://christophertkenny.com/atlas,two_party


## Backsolving 2004 on the 2012 lines from Cook's old 2012 PVI

The 2012 lines have 2008 and 2012 results but no 2004. Cook's 2012 PVI (`2012_old_cook_pvi.csv`) was computed with its older
method on those lines: 50% 2008 and 50% 2004, each relative to the national two-party Democratic share. So:

```
rel08           = dem_pct_2p_norm(2012 lines, 2008) − nat(2008)
rel04           = (cook_pvi − 0.5·rel08) / 0.5
dem_pct_2p_norm = nat(2004) + rel04
```

This PVI is published rounded to whole numbers, so the ±0.5 rounding becomes ±1 pt in the backsolved 2004 share (±0.25 pts in the
resulting 75/25 PVI). At-large states are whole states, so their backsolved value can be checked against the actual statewide
2004 result (2004 on the 2006 lines).

In [17]:
FILE_COOK_2012 = "2012_old_cook_pvi.csv"
STATE_ABBR = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR", "California": "CA", "Colorado": "CO",
    "Connecticut": "CT", "Delaware": "DE", "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA",
    "Maine": "ME", "Maryland": "MD", "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ",
    "New Mexico": "NM", "New York": "NY", "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC", "South Dakota": "SD",
    "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT", "Virginia": "VA", "Washington": "WA",
    "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY",
}


def clean_old_cook_pvi(path: Path) -> pd.DataFrame:
    """Map full state names to abbreviations and parse `2012 PVI` (`D+x` / `R+x` / `EVEN`) to a signed number."""
    cook = pd.read_csv(path, dtype=str)
    state = cook["State"].map(STATE_ABBR)
    if state.isna().any():
        raise ValueError(f"Unknown state names: {sorted(cook.loc[state.isna(), 'State'].unique())}")
    pvi = cook["2012 PVI"].replace("EVEN", "D+0").str.extract(r"^([DR])\+(\d+)$")
    if pvi.isna().any().any():
        raise ValueError(f"Unexpected 2012 PVI values: {cook.loc[pvi.isna().any(axis=1), '2012 PVI'].tolist()}")
    return cook.assign(
        State=state,
        District=cook["District"].astype(int),
        Incumbent=cook["Incumbent"].str.strip(),
        **{"2012 PVI": pvi[0].map({"D": 1, "R": -1}) * pvi[1].astype(int)},
    )


cook_2012 = clean_old_cook_pvi(RAW_DIR / FILE_COOK_2012)
cook_2012.to_csv(RAW_DIR / "2012_old_cook_pvi_cleaned.csv", index=False)

backsolved_12d_04p = backsolve_anteprior(
    cook_2012.rename(columns={"State": "state", "District": "district", "2012 PVI": "pvi"}), master, presidential,
    map_year=2012, known=2008, missing=2004, weights=PVI_WEIGHTS_50_50,
)
backsolved_12d_04p.to_csv(INTERIM_DIR / "partisan_lean_2012d_2004p.csv", index=False)

statewide_2004 = master[(master["dist_map_year"] == 2006) & (master["pres_year"] == 2004)].set_index(["state", "district"])
at_large = backsolved_12d_04p[backsolved_12d_04p["district"] == 0].set_index(["state", "district"])[["dem_pct_2p_norm"]]
at_large["actual_2004"] = statewide_2004["dem_pct_2p_norm"]
at_large["error"] = at_large["dem_pct_2p_norm"] - at_large["actual_2004"]
if (at_large["error"].abs() > 1).any():
    raise ValueError(f"At-large backsolve errors exceed PVI rounding:\n{at_large}")
at_large

,,dem_pct_2p_norm,actual_2004,error
state,district,,,
AK,0,37.487806,37.113402,0.374403
DE,0,53.793039,53.535354,0.257685
MT,0,39.581224,39.795918,-0.214694
ND,0,36.888170,36.363636,0.524534
SD,0,38.786025,38.775510,0.010515
VT,0,59.542422,60.204082,-0.661660
WY,0,29.043431,29.591837,-0.548406


## Write `partisan_lean.csv` (observed results plus the backsolved rows)

In [18]:
partisan_lean = pd.concat([master, backsolved_22d_16p, backsolved_12d_04p], ignore_index=True).sort_values(DISTRICT_ORDER, ignore_index=True)
if partisan_lean.duplicated(KEY).any():
    raise ValueError("Backsolved rows overlap observed rows")
check_districts(partisan_lean, GAPS)
partisan_lean.to_csv(INTERIM_DIR / "partisan_lean.csv", index=False)
pd.crosstab(partisan_lean["dist_map_year"], partisan_lean["pres_year"])

pres_year,2000,2004,2008,2012,2016,2020,2024
dist_map_year,,,,,,,
2006,435,435,435,0,0,0,0
2008,435,435,435,0,0,0,0
2010,435,435,435,0,0,0,0
2012,0,435,435,435,0,0,0
2014,0,0,435,435,0,0,0
2016,0,0,435,435,435,0,0
2018,0,0,417,435,435,0,0
2020,0,0,0,435,435,435,0
2022,0,0,0,0,435,435,0
